In [1]:
import os


for root, dirs, files in os.walk("/content"):
    for f in files:
        if f.lower().endswith((".xlsx", ".xls", ".csv")):
            print(os.path.join(root, f))

/content/ESG_BIG.csv
/content/MANAOS_ESG_DATA_TEST - Copie.csv
/content/sample_data/california_housing_test.csv
/content/sample_data/mnist_test.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_train.csv


In [4]:
import pandas as pd

file_path = "/content/ESG_BIG.csv"


df = pd.read_csv(file_path)

print("Shape :", df.shape)
print("\nColonnes :")
print(list(df.columns))

print("\nTypes :")
print(df.dtypes)

df.head()


Shape : (252517, 44)

Colonnes :
['INSTRUMENT_ID', 'PORTFOLIO_ID', 'PORTFOLIO_NAME', 'ISSUER_CNTRY_DOMICILE', 'NACE_GROUP_CODE', 'NACE_SECTION_CODE', 'GICS_SECTOR', 'BIO_SCORE_FLOAT', 'BIO_SCORE_INT', 'BIO_SCORE_GRADE', 'BIO_SCORE_GRADE_NUM', 'BIO_POLICY_TEXT', 'BIO_POLICY_TEXT_NUM', 'NO_NET_DEFORESTATION', 'NDPE_POLICY', 'RSPO_MEMBER', 'BIODIV_EXPOSURE_INDEX', 'NORMALIZED_EXPOSURE_0_1', 'PROTECTED_AREAS_OVERLAP_PCT', 'PROXIMITY_TO_PA_KM', 'LAND_CONVERSION_HA_SINCE_2020', 'PCT_SUPPLY_TRACEABLE', 'HCV_ASSESSMENTS_COUNT', 'IUCN_CR_SPECIES_IMPACTED', 'WATER_BOD_PROXIMITY_RISK', 'WATER_BOD_PROXIMITY_RISK_NUM', 'TNFD_READY', 'TNFD_READY_NUM', 'DISCLOSURE_QUALITY', 'DATA_QUALITY_SCORE', 'DATA_GAPS_FLAG', 'OPERATES_IN_KEY_BIOME', 'PRESSURE_SCORE', 'RESPONSE_SCORE', 'STATE_SCORE', 'WEIGHT_PRESSURE', 'WEIGHT_RESPONSE', 'WEIGHT_STATE', 'COMPOSITE_INDEX', 'BIO_SCORE_DECILE', 'HABITAT_DEPENDENCY_PCT', 'SUPPLY_FOREST_RISK_PCT', 'RESTORATION_AREA_HA', 'NATURE_RELATED_SPEND_USD_M']

Types :
INSTRUMEN

,INSTRUMENT_ID,PORTFOLIO_ID,PORTFOLIO_NAME,ISSUER_CNTRY_DOMICILE,NACE_GROUP_CODE,NACE_SECTION_CODE,GICS_SECTOR,BIO_SCORE_FLOAT,BIO_SCORE_INT,BIO_SCORE_GRADE,...,STATE_SCORE,WEIGHT_PRESSURE,WEIGHT_RESPONSE,WEIGHT_STATE,COMPOSITE_INDEX,BIO_SCORE_DECILE,HABITAT_DEPENDENCY_PCT,SUPPLY_FOREST_RISK_PCT,RESTORATION_AREA_HA,NATURE_RELATED_SPEND_USD_M
0,CHWOGHPP5054,ID618334707,Vega-Taylor,CA,49.4,H,Communication Services,5.09,51.0,D,...,43.0,0.35,0.3,0.35,58.6,6.0,33.1,22.2,0.0,22.79
1,"LUZD6HJ1JTG9,ID070312120,""Beck, Fitzgerald and...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CH0ALPL7NVA9,ID007356513,Harris LLC,KR,99.0,U,Real Estate,3.87,39.0,D,...,52.0,0.35,0.3,0.35,54.4,4.0,48.6,5.1,96.0,11.58
3,IEH91XZY96C5,ID283903159,Fowler Group,AU,5.1,B,Industrials,6.33,63.0,C,...,56.0,0.35,0.3,0.35,67.1,7.0,39.7,19.4,163.0,29.20
4,"CA3BQKH27153,ID583402803,""Bond, Rowe and Buckl...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
import numpy as np
import pandas as pd

# On part de df
df_clean = df.copy()

print("Shape initial :", df_clean.shape)

# 1) Supprimer les lignes clairement "vides" ou bizarres
#    (comme celles où quasiment tout est NaN, ou pas de portefeuille)
df_clean = df_clean.dropna(subset=["PORTFOLIO_ID", "PORTFOLIO_NAME"])

# Optionnel : si une ligne a plus de 50% de NaN, on la supprime
min_non_null = df_clean.shape[1] * 0.5
df_clean = df_clean.dropna(thresh=min_non_null)

print("Shape après drop NaN forts :", df_clean.shape)

# 2) Nettoyer les colonnes texte (strip des espaces)
str_cols = df_clean.select_dtypes(include=["object"]).columns
for col in str_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()

# 3)  uniformiser les flags Yes/No sans forcer le type booléen
bool_like_cols = [
    "NO_NET_DEFORESTATION",
    "NDPE_POLICY",
    "RSPO_MEMBER",
    "TNFD_READY",
    "DATA_GAPS_FLAG",
    "OPERATES_IN_KEY_BIOME",
]

mapping_bool_str = {
    "YES": "Yes", "Y": "Yes", "True": "Yes", "TRUE": "Yes",
    "NO": "No", "N": "No", "False": "No", "FALSE": "No",
}

for col in bool_like_cols:
    if col in df_clean.columns:
        df_clean[col] = (
            df_clean[col]
            .astype(str)
            .str.strip()
            .replace(mapping_bool_str)
        )

print("\nTypes après nettoyage :")
print(df_clean.dtypes)

df_clean.head()


Shape initial : (252517, 44)
Shape après drop NaN forts : (165938, 44)

Types après nettoyage :
INSTRUMENT_ID                     object
PORTFOLIO_ID                      object
PORTFOLIO_NAME                    object
ISSUER_CNTRY_DOMICILE             object
NACE_GROUP_CODE                  float64
NACE_SECTION_CODE                 object
GICS_SECTOR                       object
BIO_SCORE_FLOAT                  float64
BIO_SCORE_INT                    float64
BIO_SCORE_GRADE                   object
BIO_SCORE_GRADE_NUM              float64
BIO_POLICY_TEXT                   object
BIO_POLICY_TEXT_NUM              float64
NO_NET_DEFORESTATION              object
NDPE_POLICY                       object
RSPO_MEMBER                       object
BIODIV_EXPOSURE_INDEX            float64
NORMALIZED_EXPOSURE_0_1          float64
PROTECTED_AREAS_OVERLAP_PCT      float64
PROXIMITY_TO_PA_KM               float64
LAND_CONVERSION_HA_SINCE_2020    float64
PCT_SUPPLY_TRACEABLE             float64
HC

,INSTRUMENT_ID,PORTFOLIO_ID,PORTFOLIO_NAME,ISSUER_CNTRY_DOMICILE,NACE_GROUP_CODE,NACE_SECTION_CODE,GICS_SECTOR,BIO_SCORE_FLOAT,BIO_SCORE_INT,BIO_SCORE_GRADE,...,STATE_SCORE,WEIGHT_PRESSURE,WEIGHT_RESPONSE,WEIGHT_STATE,COMPOSITE_INDEX,BIO_SCORE_DECILE,HABITAT_DEPENDENCY_PCT,SUPPLY_FOREST_RISK_PCT,RESTORATION_AREA_HA,NATURE_RELATED_SPEND_USD_M
0,CHWOGHPP5054,ID618334707,Vega-Taylor,CA,49.4,H,Communication Services,5.09,51.0,D,...,43.0,0.35,0.3,0.35,58.6,6.0,33.1,22.2,0.0,22.79
2,CH0ALPL7NVA9,ID007356513,Harris LLC,KR,99.0,U,Real Estate,3.87,39.0,D,...,52.0,0.35,0.3,0.35,54.4,4.0,48.6,5.1,96.0,11.58
3,IEH91XZY96C5,ID283903159,Fowler Group,AU,5.1,B,Industrials,6.33,63.0,C,...,56.0,0.35,0.3,0.35,67.1,7.0,39.7,19.4,163.0,29.20
5,KRFXB4J3HCA4,ID278513589,Powell and Sons,SE,78.1,N,Communication Services,6.20,62.0,C,...,68.0,0.35,0.3,0.35,75.5,7.0,38.1,53.3,236.0,28.39
6,CHL4X5O8GTD3,ID278513589,Powell and Sons,SE,78.1,N,Communication Services,5.29,53.0,D,...,64.0,0.35,0.3,0.35,65.5,6.0,78.3,35.0,49.0,22.27


In [7]:
clean_path = "/content/clean_esg.csv"
sample_path = "/content/sample_esg.csv"

print("Shape du df_clean final :", df_clean.shape)

# 1) Dataset complet nettoyé
df_clean.to_csv(clean_path, index=False)
print(f"Fichier complet sauvegardé : {clean_path}")

# 2) Sample pour les tests (par ex. 50 000 lignes max)
n_sample = min(50000, len(df_clean))
df_sample = df_clean.sample(n=n_sample, random_state=42)

df_sample.to_csv(sample_path, index=False)
print(f"Sample ({n_sample} lignes) sauvegardé : {sample_path}")


Shape du df_clean final : (165938, 44)
Fichier complet sauvegardé : /content/clean_esg.csv
Sample (50000 lignes) sauvegardé : /content/sample_esg.csv
